In [ ]:

                                # Milestone 2 Enhancement:
                                ## Set to modern Dash
from dash import Dash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
                                # Milestone 2 Enhancement:
                                ## infer_jupyter_proxy_config() was removed for local testing because it caused
                                ## compatibility issues in this notebook environment.
                                ## If you are running in Codio environment, uncomment this next line:
## JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import json

from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

                                # Milestone 2 Enhancement:
                                ## Removed hard-coded credentials and replaced with environment variables

### IMPORTANT DEVELOPER NOTE ###
## SEE README FOR INSTRUCTIONS ON HOW TO TEST THIS PROGRAM ##

username = os.getenv("AAC_USERNAME")
password = os.getenv("AAC_PASSWORD")

# Fallback to local config file for development/demo purposes
if not username or not password:
    try:
        with open("db_config.json", "r") as file:
            config = json.load(file)
            username = config.get("username")
            password = config.get("password")
    except FileNotFoundError:
        raise ValueError("No environment variables or config file found for database credentials.")

if not username or not password:
    raise ValueError("Database credentials are missing from both environment variables and config file.")

# Connect to database via CRUD Module

# Milestone 2 Enhancement:
## Database connection now uses secure credentials loaded from environment variables

db = AnimalShelter(username, password)
# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned

                                # Milestone 4 Enhancement:
                                ## Create indexes at application startup to improve performance for frequently used dashboard queries
db.ensure_indexes()
                                # Milestone 4 Enhancement:
                                ##Defines a projection so the dashboard only retrieves fields required for the table, chart, and map
DASHBOARD_PROJECTION = {
    "_id": 0,
    "name": 1,
    "animal_type": 1,
    "breed": 1,
    "color": 1,
    "sex_upon_outcome": 1,
    "age_upon_outcome_in_weeks": 1,
    "location_lat": 1,
    "location_long": 1
}

                                # Milestone 4 Enhancement:
                                ## Limits the initial query size to improve startup performance and reduce unnecessary memory usage
INITIAL_LOAD_LIMIT = 1000

### Database TEST: Remove or Comment out before release
test_records = db.read({}, projection=DASHBOARD_PROJECTION, limit=INITIAL_LOAD_LIMIT)
print("Raw database test returned:", len(test_records), "records")

df = pd.DataFrame.from_records(
    db.read({}, projection=DASHBOARD_PROJECTION, limit=INITIAL_LOAD_LIMIT)
)

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)

# Milestone 2 Enhancement:
##Added safe check before dropping '_id' column to prevent runtime errors

if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)

#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Div([
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '100px', 'marginRight': '20px'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        ),
        html.Div([
            html.H1('CS-340 Dashboard', style={'margin': '0'}),
            html.H4('Ryan Blackburn Project 2', style={'margin': '0'})  # <-- unique identifier
        ])
    ], style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center'}),
    
    html.Hr(),
    html.Div([
        
     
# Interactive filtering options.
        html.Label("Rescue Type Filter:"),
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'Water Rescue', 'value': 'water'},
            {'label': 'Mountain / Wilderness Rescue', 'value': 'mountain'},
            {'label': 'Disaster / Individual Tracking', 'value': 'disaster'},
            {'label': 'Reset', 'value': 'reset'}
        ],
        value='reset',
        inline=True
    )
]),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         sort_action="native",
                         filter_action="native",
                         page_action="native",
                         page_current=0,
                         page_size=10,
                         row_selectable="single",
                         selected_rows=[0],
                         selected_columns=[],
                         style_table={'overflowX': 'auto'},
                         style_cell={'textAlign': 'left', 'padding': '5px'},
                         style_header={'fontWeight': 'bold'},
                         ),
    html.Br(),
    html.Hr(),

#Dashboard config so that chart and geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    [Output('datatable-id', 'data'),
     Output('datatable-id', 'columns')],
    [Input('filter-type', 'value')]
)

# Milestone 2 Enhancement:
## Moved rescue query logic out of this callback and into the CRUD module
### Improves separation of concerns and keeps UI logic independent from database logic

                                    # Milestone 4 Enhancement:
                                    ##Applies filtered retrieval through the CRUD module to reduce unnecessary data transfer
def update_dashboard(filter_type):
    records = db.read_by_rescue_type(
        filter_type,
        projection=DASHBOARD_PROJECTION
    )
    dff = pd.DataFrame.from_records(records)

    if not dff.empty and '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    columns = [{"name": i, "id": i, "deletable": False, "selectable": True} for i in dff.columns]
    data = dff.to_dict('records')

    return data, columns

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
                                                                                           
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

                                # Milestone 2 Enhancement:
                                ## Added validation to ensure required data exists before generating visualization

    if dff.empty or 'breed' not in dff.columns:
        return [dcc.Graph(figure=px.pie(title='Preferred Animals'))]

    breed_counts = dff['breed'].value_counts().nlargest(10).reset_index()
    breed_counts.columns = ['breed', 'count']

    fig = px.pie(breed_counts, names='breed', values='count', title='Preferred Animals')

    return [
        dcc.Graph(
            figure = fig
        )
    ]
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable

@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
                                # Milestone 2 Enhancement:
                                ## Improved map reliability by validating input data, selected row, and coordinates
                                ### Also replaced fragile positional indexing with named column access
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    required_columns = {"location_lat", "location_long", "breed", "name"}
    if not required_columns.issubset(dff.columns):
        return []

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    if row >= len(dff):
        row = 0

    selected_row = dff.iloc[row]

                                # Milestone 2 Enhancement:
                                ## Convert coordinates safely and handle invalid values to prevent runtime errors
    lat = pd.to_numeric(selected_row["location_lat"], errors="coerce")
    lon = pd.to_numeric(selected_row["location_long"], errors="coerce")

    if pd.isna(lat) or pd.isna(lon):
        return []

    animal_breed = selected_row.get("breed", "Unknown Breed")
    animal_name = selected_row.get("name", "Unnamed Animal")

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[lat, lon],
            zoom=12,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(str(animal_breed)),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(str(animal_name))
                        ])
                    ]
                )
            ]
        )
    ]                                                  

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run(debug=False)

Raw database test returned: 10000 records
